# Generación de Embeddings e Índice FAISS con BAAI/bge-m3 en GPU gratuita (Google Colab)

Este notebook genera los embeddings de **BAAI/bge-m3** (1024 dimensiones) para los 30.287 documentos del corpus de manuales y construye el índice FAISS `IndexFlatIP`.

**Tiempo estimado con GPU T4:** ~2 a 3 minutos.

### 1. Verificar GPU activa
Asegúrate de que en el menú superior esté seleccionado: **Entorno de ejecución $\rightarrow$ Cambiar tipo de entorno de ejecución $\rightarrow$ T4 GPU**.

In [ ]:
!nvidia-smi

### 2. Instalar dependencias necesarias

In [ ]:
!pip install -q sentence-transformers faiss-cpu pyyaml

### 3. Obtener el proyecto
Elige **Opción A** (clonar desde tu repositorio de GitHub) o **Opción B** (subir directamente el archivo `corpus.tar.gz`).

In [ ]:
# Opción A: Si tu repo es público o usas token:
# !git clone -b feat/multilingual-bge-m3 https://github.com/TU_USUARIO/TU_REPO.git code
# %cd code

# Opción B: Subir corpus.tar.gz manualmente usando el botón de archivos o este widget:
from google.colab import files
import os

if not os.path.exists('dataset_builder/corpus'):
    print('Por favor, sube corpus.tar.gz:')
    uploaded = files.upload()
    !mkdir -p dataset_builder
    !tar -xzf corpus.tar.gz -C dataset_builder/

### 4. Generar embeddings con BAAI/bge-m3 en GPU

In [ ]:
!python dataset_builder/embedder/bge-m3/generate_embeddings.py \
    --corpus dataset_builder/corpus \
    --output dataset_builder/embedder/artifacts \
    --vector-name bge_m3_vectors.npz \
    --model BAAI/bge-m3 \
    --batch-size 64 \
    --device cuda

### 5. Construir índice FAISS (1024 dimensiones)

In [ ]:
!python dataset_builder/indexer/bge-m3/build_index.py \
    --vectors dataset_builder/embedder/artifacts/bge_m3_vectors.npz \
    --output dataset_builder/indexer/artifacts/bge-m3

### 6. Empaquetar y descargar los artefactos generados

In [ ]:
!tar -czvf bge_m3_artifacts.tar.gz \
    dataset_builder/embedder/artifacts/bge_m3_vectors.npz \
    dataset_builder/indexer/artifacts/bge-m3/index.faiss \
    dataset_builder/indexer/artifacts/bge-m3/aliases.npy

from google.colab import files
files.download('bge_m3_artifacts.tar.gz')